# RAG Ingestion with Ray Data

Ingest PDFs into Milvus for RAG using a 3-stage Ray Data streaming pipeline:
**Docling** (parse + chunk) → **vLLM** (embed via `vLLMEngineProcessorConfig`) → **Milvus** (store).

The pipeline runs as a **RayJob** on a **RayCluster** deployed via
`manifests/raycluster-rag-optimized.yaml` (see **Connect to RayCluster** below).
Ray Data streams the stages so they overlap — embedding starts before all PDFs
finish parsing. The pipeline code lives in `docling_milvus_process.py`.

### Prerequisites

- **Red Hat OpenShift AI** with the KubeRay operator installed
- **Milvus** accessible from the Ray cluster
- PDFs uploaded to a **ReadWriteMany PVC** mounted on all Ray nodes
- A workbench with `codeflare-sdk` installed (this notebook)
- A **RayCluster** with CPU and GPU workers — review and apply `manifests/raycluster-rag-optimized.yaml`, tuning resource values for your cluster (see README)

In [ ]:
%pip install -q codeflare-sdk pymilvus

In [ ]:
import os
import shutil
import tempfile
from pathlib import Path

from codeflare_sdk import RayJob, get_cluster

## Authenticate

Log in to your OpenShift cluster. Replace the command below with your own
`oc login` invocation (token + server URL), or skip this cell if you are
already authenticated via the workbench.

In [ ]:
# Paste your own oc login command here:
# !oc login --token=<your-token> --server=<your-api-url>

!oc whoami

## Configure

> ⚠️ **You MUST update the "Required" values below to match your cluster.**

In [ ]:
# ============================================================
# REQUIRED — update these for your cluster
# ============================================================
EXISTING_CLUSTER_NAME = "raytest"
NAMESPACE = "rag-example"
PVC_CLAIM_NAME = "rag-data-pvc"  # ReadWriteMany; must exist in NAMESPACE
MILVUS_HOST = "milvus-milvus.milvus.svc.cluster.local"

# ⚠️ WARNING: When True, the pipeline DROPS and recreates the Milvus collection, destroying all existing data.
# Set to False to fail gracefully if the collection already exists.
DROP_EXISTING_COLLECTION = "true"

# ============================================================
# OPTIONAL — safe defaults, tune for your hardware
# ============================================================

# Storage (PVC mounted on all Ray workers)
PVC_MOUNT_PATH = "/mnt/data"
INPUT_PATH = "input/pdfs"

# Milvus
MILVUS_PORT = "19530"
MILVUS_DB = "default"
MILVUS_COLLECTION = "rag_documents"
MILVUS_TEXT_MAX_CHARS = "8192"

# vLLM embedding (GPU)
VLLM_MODEL_SOURCE = "intfloat/multilingual-e5-large"
VLLM_BATCH_SIZE = "4"
VLLM_CONCURRENCY = "1"
EMBEDDING_DIM = "1024"
VLLM_ENGINE_KWARGS_JSON = '{"enforce_eager":true,"gpu_memory_utilization":0.8}'
VLLM_ACCELERATOR_TYPE = ""  # optional Ray accelerator type for GPU workers

# Pipeline tuning
NUM_FILES = "0"  # PDFs to process (0 = all)
# 70% rule: max_actors ≈ floor(total_cluster_cpus × 0.70 / CPUS_PER_ACTOR)
NUM_ACTORS = "6"  # Docling parsing actors — see README "Sizing for your hardware"
CPUS_PER_ACTOR = "4"  # CPUs per Docling actor
NUM_MILVUS_ACTORS = "2"
BATCH_SIZE = "2"  # PDFs per actor batch
MILVUS_BATCH_SIZE = "64"
REPARTITION_FACTOR = "2"
CHUNK_MAX_TOKENS = "256"

print(f"Cluster:    {EXISTING_CLUSTER_NAME} in {NAMESPACE}")
print(f"Input:      {PVC_MOUNT_PATH}/{INPUT_PATH}")
print(f"Milvus:     {MILVUS_HOST}:{MILVUS_PORT}/{MILVUS_DB}.{MILVUS_COLLECTION}")
print(f"Embedding:  {VLLM_MODEL_SOURCE} (dim={EMBEDDING_DIM})")
print(f"GPU:        {VLLM_ENGINE_KWARGS_JSON}")
print(
    f"Actors:     {NUM_ACTORS} docling x {CPUS_PER_ACTOR} CPUs, {NUM_MILVUS_ACTORS} milvus"
)
print(f"Files:      {'ALL' if NUM_FILES == '0' else NUM_FILES}")

## Connect to RayCluster

Attach to the RayCluster created via `oc apply` of the manifest and retrieve the dashboard URL.
See the Setup section in the README for instructions on reviewing and applying
`manifests/raycluster-rag-optimized.yaml`.

In [ ]:
cluster = get_cluster(EXISTING_CLUSTER_NAME, namespace=NAMESPACE, verify_tls=False)

print(f"Dashboard: {cluster.cluster_dashboard_uri()}")

### Pipeline overview

The `_run_pipeline` function chains three stages using Ray Data streaming
execution:

```python
# Stage 1: Parse + chunk (CPU-heavy, bottleneck)
ds = ds.map_batches(
    DoclingChunkActor,
    concurrency=NUM_ACTORS,
    batch_size=BATCH_SIZE,
    num_cpus=CPUS_PER_ACTOR,
)

# Stage 2: Embed with vLLM (GPU, via Ray Data LLM processor)
embed_processor = _build_vllm_embed_processor()
ds = embed_processor(ds)

# Stage 3: Write to Milvus (I/O-bound)
results = ds.map_batches(
    MilvusWriteActor,
    concurrency=NUM_MILVUS_ACTORS,
    batch_size=MILVUS_BATCH_SIZE,
    num_cpus=1,
)
```

Ray Data's streaming executor overlaps stages: downstream actors start as
soon as upstream blocks are ready.

## Submit RayJob

In [ ]:
pipeline_env = {
    "PVC_MOUNT_PATH": PVC_MOUNT_PATH,
    "INPUT_PATH": INPUT_PATH,
    "MILVUS_HOST": MILVUS_HOST,
    "MILVUS_PORT": MILVUS_PORT,
    "MILVUS_DB": MILVUS_DB,
    "MILVUS_COLLECTION": MILVUS_COLLECTION,
    "DROP_EXISTING_COLLECTION": DROP_EXISTING_COLLECTION,
    "MILVUS_TEXT_MAX_CHARS": MILVUS_TEXT_MAX_CHARS,
    "VLLM_MODEL_SOURCE": VLLM_MODEL_SOURCE,
    "VLLM_BATCH_SIZE": VLLM_BATCH_SIZE,
    "VLLM_CONCURRENCY": VLLM_CONCURRENCY,
    "VLLM_ENGINE_KWARGS_JSON": VLLM_ENGINE_KWARGS_JSON,
    "VLLM_ACCELERATOR_TYPE": VLLM_ACCELERATOR_TYPE,
    "EMBEDDING_DIM": EMBEDDING_DIM,
    "NUM_FILES": NUM_FILES,
    "NUM_ACTORS": NUM_ACTORS,
    "CPUS_PER_ACTOR": CPUS_PER_ACTOR,
    "NUM_MILVUS_ACTORS": NUM_MILVUS_ACTORS,
    "BATCH_SIZE": BATCH_SIZE,
    "MILVUS_BATCH_SIZE": MILVUS_BATCH_SIZE,
    "REPARTITION_FACTOR": REPARTITION_FACTOR,
    "CHUNK_MAX_TOKENS": CHUNK_MAX_TOKENS,
    "HF_HOME": "/mnt/data/huggingface",
    "XDG_CACHE_HOME": "/tmp/cache",
}

In [ ]:
from datetime import UTC, datetime

job_name = f"rag-ingest-{datetime.now(UTC):%Y%m%d-%H%M%S}"

notebook_dir = Path(".").resolve()
slim_dir = tempfile.mkdtemp(prefix="rag-job-")
shutil.copy2(notebook_dir / "docling_milvus_process.py", slim_dir)
print(f"Working dir: {slim_dir} ({os.listdir(slim_dir)})")

job = RayJob(
    job_name=job_name,
    cluster_name=EXISTING_CLUSTER_NAME,
    namespace=NAMESPACE,
    entrypoint="python docling_milvus_process.py",
    runtime_env={
        "working_dir": slim_dir,
        "env_vars": pipeline_env,
    },
    active_deadline_seconds=10800,
)

job.submit()
print(f"RayJob '{job.name}' submitted.")

## Monitor

In [ ]:
job.status()

In [ ]:
!oc get pods -n {NAMESPACE} -l ray.io/cluster={EXISTING_CLUSTER_NAME}

In [ ]:
# Fetch job logs (run after the job completes for the performance report)
!oc logs -l job-name={job.name} -n {NAMESPACE} --tail=100

## Verify

Once the job completes, confirm that vectors were inserted into Milvus.
This connects directly from the workbench — no Ray cluster needed.

In [ ]:
from pymilvus import MilvusClient

client = MilvusClient(uri=f"http://{MILVUS_HOST}:{MILVUS_PORT}", db_name=MILVUS_DB)
client.load_collection(MILVUS_COLLECTION)
stats = client.get_collection_stats(MILVUS_COLLECTION)
row_count = stats.get("row_count", stats.get("data", {}).get("row_count", "?"))
print(f"Collection '{MILVUS_COLLECTION}': {row_count} vectors")
print("\nReady for querying — open rag_query.ipynb to test retrieval.")